## DAY 24 TIME BASED BUSINESS ANALYSIS

In [84]:
import pandas as pd
import numpy as np

In [54]:
df = pd.read_csv("Data_Co_Supply_Chain_Dataset.csv" ,encoding="latin1")
df[[
          "order date (DateOrders)",
          "shipping date (DateOrders)"
]].dtypes

order date (DateOrders)       object
shipping date (DateOrders)    object
dtype: object

In [55]:
df[[
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]].head()

,order date (DateOrders),shipping date (DateOrders)
0,1/31/2018 22:56,02-03-2018 22:56
1,1/13/2018 12:27,1/18/2018 12:27
2,1/13/2018 12:06,1/17/2018 12:06
3,1/13/2018 11:45,1/16/2018 11:45
4,1/13/2018 11:24,1/15/2018 11:24


In [56]:
df["order date (DateOrders)"] = pd.to_datetime(
     df["order date (DateOrders)"] , format="mixed"
)
df["shipping date (DateOrders)"] = pd.to_datetime(
     df["shipping date (DateOrders)"] , format="mixed"
)


In [57]:
df[[
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]].dtypes

order date (DateOrders)       datetime64[ns]
shipping date (DateOrders)    datetime64[ns]
dtype: object

In [58]:
df[[
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]].head()

,order date (DateOrders),shipping date (DateOrders)
0,2018-01-31 22:56:00,2018-02-03 22:56:00
1,2018-01-13 12:27:00,2018-01-18 12:27:00
2,2018-01-13 12:06:00,2018-01-17 12:06:00
3,2018-01-13 11:45:00,2018-01-16 11:45:00
4,2018-01-13 11:24:00,2018-01-15 11:24:00


### Create Time Features

In [59]:
df["Order_Year"] = df["order date (DateOrders)"].dt.year
df["Order_Month"] = df["order date (DateOrders)"].dt.month
df["Order_Month_Name"] = df["order date (DateOrders)"].dt.month_name()

In [60]:
df[["Order_Year" , "Order_Month" , "Order_Month_Name"]].head()

,Order_Year,Order_Month,Order_Month_Name
0,2018,1,January
1,2018,1,January
2,2018,1,January
3,2018,1,January
4,2018,1,January


### Monthly Business Performance

In [61]:
monthly_analysis = df.groupby(
     ["Order_Year" , "Order_Month" , "Order_Month_Name"]
).agg(
     Total_sales = ("Sales" , "sum"),
     Total_profit = ("Order Profit Per Order" , "sum"),
     Total_orders = ("Order Id" , "nunique")
)
monthly_analysis.sort_values(
     by="Total_sales", ascending=False
).head(10)

Total_sales   Total_profit  \
Order_Year Order_Month Order_Month_Name                                
2017       9           September         1.143775e+06  122462.390153   
           8           August            1.109337e+06  131501.160211   
           5           May               1.105485e+06  115014.640014   
           7           July              1.104373e+06  113026.700038   
           10          October           1.073994e+06  113447.169883   
2015       12          December          1.057841e+06  110641.549881   
           1           January           1.051590e+06  111660.740132   
           3           March             1.051254e+06  113778.210191   
           5           May               1.050478e+06  112147.900143   
           10          October           1.049154e+06  101757.870040   

                                         Total_orders  
Order_Year Order_Month Order_Month_Name                
2017       9           September                 1723  
           8           August                    1768  
           5           May                       1763  
           7           July                      1776  
           10          October                   2101  
2015       12          December                  1805  
           1           January                   1787  
           3           March                     1781  
           5           May                       1776  
           10          October                   1775

### Month-over-Month Analysis

In [62]:
# we need to create proper chronological table 
monthly_trend =( df.groupby(
     ["Order_Year" , "Order_Month" , "Order_Month_Name"]
).agg(
     Total_sales = ("Sales" , "sum"),
     Total_profit = ("Order Profit Per Order" , "sum") ,
     Total_orders = ("Order Id" , "nunique")
).reset_index().sort_values(
     ["Order_Year" , "Order_Month"]
)
)
monthly_trend.head(10)

,Order_Year,Order_Month,Order_Month_Name,Total_sales,Total_profit,Total_orders
0,2015,1,January,1.051590e+06,111660.740132,1787
1,2015,2,February,9.270099e+05,99140.660196,1585
2,2015,3,March,1.051254e+06,113778.210191,1781
3,2015,4,April,1.014463e+06,108083.679957,1710
4,2015,5,May,1.050478e+06,112147.900143,1776
5,2015,6,June,1.024006e+06,110147.160313,1725
6,2015,7,July,1.038081e+06,115624.059879,1763
7,2015,8,August,1.029495e+06,117979.770302,1762
8,2015,9,September,1.018339e+06,113467.940118,1706
9,2015,10,October,1.049154e+06,101757.870040,1775


In [63]:
## shift() : compare the current month with the previous month
monthly_trend["Previous_Month_Sales"] = (
    monthly_trend["Total_sales"].shift(1)
)

In [64]:
monthly_trend["Sales_MoM_change_%"] = (
     (
          monthly_trend["Total_sales"] -
          monthly_trend["Previous_Month_Sales"]
     )
     .div(monthly_trend["Previous_Month_Sales"]).mul(100)
)

In [65]:
monthly_trend.head(10)

,Order_Year,Order_Month,Order_Month_Name,Total_sales,Total_profit,Total_orders,Previous_Month_Sales,Sales_MoM_change_%
0,2015,1,January,1.051590e+06,111660.740132,1787,NaN,NaN
1,2015,2,February,9.270099e+05,99140.660196,1585,1.051590e+06,-11.846839
2,2015,3,March,1.051254e+06,113778.210191,1781,9.270099e+05,13.402639
3,2015,4,April,1.014463e+06,108083.679957,1710,1.051254e+06,-3.499670
4,2015,5,May,1.050478e+06,112147.900143,1776,1.014463e+06,3.550169
5,2015,6,June,1.024006e+06,110147.160313,1725,1.050478e+06,-2.520020
6,2015,7,July,1.038081e+06,115624.059879,1763,1.024006e+06,1.374505
7,2015,8,August,1.029495e+06,117979.770302,1762,1.038081e+06,-0.827151
8,2015,9,September,1.018339e+06,113467.940118,1706,1.029495e+06,-1.083647
9,2015,10,October,1.049154e+06,101757.870040,1775,1.018339e+06,3.026073


In [66]:
# the biggest increase
monthly_trend.loc[
     monthly_trend["Sales_MoM_change_%"].idxmax()
]

Order_Year                        2015
Order_Month                          3
Order_Month_Name                 March
Total_sales             1051253.690334
Total_profit             113778.210191
Total_orders                      1781
Previous_Month_Sales     927009.898168
Sales_MoM_change_%           13.402639
Name: 2, dtype: object

In [67]:
# the biggest decrease
monthly_trend.loc[
     monthly_trend["Sales_MoM_change_%"].idxmin()
]

Order_Year                        2017
Order_Month                         11
Order_Month_Name              November
Total_sales              626914.383667
Total_profit              67791.250205
Total_orders                      2055
Previous_Month_Sales    1073994.169021
Sales_MoM_change_%          -41.627767
Name: 34, dtype: object

### Shipping Duration Analysis

In [68]:
df["Shipping_Days"] = (
    df["shipping date (DateOrders)"]
    - df["order date (DateOrders)"]
).dt.days

In [69]:
df[[
    "Order Id",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Shipping_Days"
]].head(10)

,Order Id,order date (DateOrders),shipping date (DateOrders),Shipping_Days
0,77202,2018-01-31 22:56:00,2018-02-03 22:56:00,3
1,75939,2018-01-13 12:27:00,2018-01-18 12:27:00,5
2,75938,2018-01-13 12:06:00,2018-01-17 12:06:00,4
3,75937,2018-01-13 11:45:00,2018-01-16 11:45:00,3
4,75936,2018-01-13 11:24:00,2018-01-15 11:24:00,2
5,75935,2018-01-13 11:03:00,2018-01-19 11:03:00,6
6,75934,2018-01-13 10:42:00,2018-01-15 10:42:00,2
7,75933,2018-01-13 10:21:00,2018-01-15 10:21:00,2
8,75932,2018-01-13 10:00:00,2018-01-16 10:00:00,3
9,75931,2018-01-13 09:39:00,2018-01-15 09:39:00,2


In [70]:
df["Shipping_Days"].describe()

count    180519.000000
mean          3.471856
std           1.670471
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: Shipping_Days, dtype: float64

### Shipping Performance by Region

In [71]:
region_shipping = df.groupby("Order Region").agg(
     Average_Shipping_Days = ("Shipping_Days" , "mean"),
     Median_Shipping_Days = ("Shipping_Days" , "median"),
     Total_Orders = ("Order Id" , "nunique")
)


In [72]:
region_shipping = region_shipping.sort_values(
     by="Average_Shipping_Days",
     ascending=False
)

In [73]:
region_shipping.head(10)

,Average_Shipping_Days,Median_Shipping_Days,Total_Orders
Order Region,,,
Central Africa,3.546213,3.0,556
West Africa,3.510552,3.0,1223
Eastern Asia,3.500275,3.0,3318
East Africa,3.492441,3.0,613
Central America,3.486010,3.0,9396
Eastern Europe,3.484439,3.0,1292
Northern Europe,3.479984,3.0,3716
West of USA,3.478794,3.0,2667
Caribbean,3.478240,3.0,2806


### Shipping Duration × Delivery Risk

In [74]:
delivery_risk_shipping = df.groupby("Late_delivery_risk").agg(
    Average_Shipping_Days=("Shipping_Days", "mean"),
    Median_Shipping_Days=("Shipping_Days", "median"),
    Total_Orders=("Order Id", "nunique")
)

In [75]:
delivery_risk_shipping

,Average_Shipping_Days,Median_Shipping_Days,Total_Orders
Late_delivery_risk,,,
0,2.777072,3.0,29704
1,4.044253,5.0,36048


### Delivery Status Analysis

In [76]:
delivery_status_analysis = df.groupby("Delivery Status").agg(
    Average_Shipping_Days=("Shipping_Days", "mean"),
    Median_Shipping_Days=("Shipping_Days", "median"),
    Total_Orders=("Order Id", "nunique")
)

In [77]:
delivery_status_analysis.sort_values(
    by="Average_Shipping_Days",
    ascending=False
)

,Average_Shipping_Days,Median_Shipping_Days,Total_Orders
Delivery Status,,,
Late delivery,4.044253,5.0,36048
Shipping canceled,3.450477,3.0,2855
Shipping on time,2.975214,4.0,11722
Advance shipping,2.498149,2.0,15127


### Regional Delivery Risk

In [78]:
regional_risk = df.groupby("Order Region").agg(
    Late_Risk_Rate=("Late_delivery_risk", "mean"),
    Average_Shipping_Days=("Shipping_Days", "mean"),
    Total_Orders=("Order Id", "nunique")
)

In [79]:
regional_risk["Late_Risk_Rate"] = (
    regional_risk["Late_Risk_Rate"] * 100
)

In [80]:
regional_risk.sort_values(
     by="Late_Risk_Rate",
     ascending=False
).head(10)

,Late_Risk_Rate,Average_Shipping_Days,Total_Orders
Order Region,,,
Central Africa,57.960644,3.546213,556
South Asia,56.266977,3.476652,3335
East Africa,55.939525,3.492441,613
Western Europe,55.848611,3.471467,10010
South of USA,55.772559,3.467985,1345
Eastern Europe,55.663265,3.484439,1292
East of USA,55.661605,3.471728,2323
Southeast Asia,55.529930,3.476780,4356
Central Asia,55.334539,3.394213,184


### Find the Highest-Risk Region

In [81]:
highest_risk_region = regional_risk["Late_Risk_Rate"].idxmax()

regional_risk.loc[highest_risk_region]

Late_Risk_Rate            57.960644
Average_Shipping_Days      3.546213
Total_Orders             556.000000
Name: Central Africa, dtype: float64

In [82]:
slowest_region = regional_risk["Average_Shipping_Days"].idxmax()

regional_risk.loc[slowest_region]

Late_Risk_Rate            57.960644
Average_Shipping_Days      3.546213
Total_Orders             556.000000
Name: Central Africa, dtype: float64

### Create an Operational Performance Flag

In [85]:
conditions = [
    regional_risk["Late_Risk_Rate"] >= 55,
    regional_risk["Late_Risk_Rate"] >= 50
]

choices = [
    "High Operational Risk",
    "Moderate Operational Risk"
]

regional_risk["Operational_Risk_Category"] = np.select(
    conditions,
    choices,
    default="Lower Operational Risk"
)

In [86]:
regional_risk

,Late_Risk_Rate,Average_Shipping_Days,Total_Orders,Operational_Risk_Category
Order Region,,,,
Canada,48.800834,3.320125,309,Lower Operational Risk
Caribbean,53.077663,3.478240,2806,Moderate Operational Risk
Central Africa,57.960644,3.546213,556,High Operational Risk
Central America,54.754596,3.486010,9396,Moderate Operational Risk
Central Asia,55.334539,3.394213,184,High Operational Risk
East Africa,55.939525,3.492441,613,High Operational Risk
East of USA,55.661605,3.471728,2323,High Operational Risk
Eastern Asia,54.326923,3.500275,3318,Moderate Operational Risk
Eastern Europe,55.663265,3.484439,1292,High Operational Risk


# 💼 Business Insights
### Sales Trends
September 2017 had the highest sales in the displayed monthly ranking.
March 2015 recorded the largest observed MoM sales increase.
November 2017 recorded the largest observed MoM sales decrease.
### Operational Performance
Average shipping duration was approximately 3.47 days.
Median shipping duration was 3 days.
Late-risk transactions had an average shipping duration of 4.04 days, compared with 2.78 days for no-risk transactions.
### Regional Performance
Central Africa had the highest observed late-risk rate at approximately 57.96%.
Central Africa also had the highest observed average shipping duration in the regional analysis.
### Analytical Caution

The analysis identifies relationships and patterns in the dataset.